In [5]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
import joblib


In [6]:
df1 = pd.read_csv("dataset1.csv")

In [7]:
print(df1.columns)


Index(['title', 'magnitude', 'date_time', 'cdi', 'mmi', 'alert', 'tsunami',
       'sig', 'net', 'nst', 'dmin', 'gap', 'magType', 'depth', 'latitude',
       'longitude', 'location', 'continent', 'country'],
      dtype='object')


In [16]:
df_scaled = df1.copy() 

for column in df_scaled.select_dtypes(include=np.number).columns: 
    df_scaled[column] = (df_scaled[column] - df_scaled[column].min()) / (df_scaled[column].max() - df_scaled[column].min())     
  

print(df_scaled)

                                         title  magnitude         date_time  \
0             M 6.5 - 42 km W of Sola, Vanuatu   0.000000  16-08-2023 12:47   
1     M 6.5 - 43 km S of Intipucá, El Salvador   0.000000  19-07-2023 00:22   
2     M 6.6 - 25 km ESE of Loncopué, Argentina   0.038462  17-07-2023 03:05   
3        M 7.2 - 98 km S of Sand Point, Alaska   0.269231  16-07-2023 06:48   
4                     M 7.3 - Alaska Peninsula   0.307692  16-07-2023 06:48   
..                                         ...        ...               ...   
995          M 7.1 - 85 km S of Tungor, Russia   0.230769  27-05-1995 13:03   
996     M 7.7 - 249 km E of Vao, New Caledonia   0.461538  16-05-1995 20:12   
997  M 6.9 - 27 km NNW of Maubara, Timor Leste   0.153846  14-05-1995 11:33   
998           M 6.6 - 10 km W of Aianí, Greece   0.038462  13-05-1995 08:47   
999  M 7.1 - 14 km NE of Cabatuan, Philippines   0.230769  05-05-1995 03:53   

          cdi       mmi   alert  tsunami       sig 

In [17]:
df_numeric = df1.select_dtypes(include=[np.number])
corr = df_numeric.corr()
corr.style.background_gradient(cmap='coolwarm').format("{:.2f}")

,magnitude,cdi,mmi,tsunami,sig,nst,dmin,gap,depth,latitude,longitude
magnitude,1.00,0.16,0.28,-0.00,0.51,0.09,-0.08,-0.09,0.02,-0.02,-0.01
cdi,0.16,1.00,0.20,0.29,0.48,-0.00,0.14,0.28,-0.05,0.07,-0.13
mmi,0.28,0.20,1.00,-0.18,0.40,0.11,-0.31,-0.07,-0.52,0.18,-0.02
tsunami,-0.00,0.29,-0.18,1.00,0.02,-0.43,0.44,0.24,0.07,-0.13,-0.13
sig,0.51,0.48,0.40,0.02,1.00,0.02,-0.05,0.16,-0.08,0.18,-0.16
nst,0.09,-0.00,0.11,-0.43,0.02,1.00,-0.40,0.00,-0.10,0.11,0.16
dmin,-0.08,0.14,-0.31,0.44,-0.05,-0.40,1.00,0.09,0.19,-0.24,-0.08
gap,-0.09,0.28,-0.07,0.24,0.16,0.00,0.09,1.00,-0.06,0.05,-0.27
depth,0.02,-0.05,-0.52,0.07,-0.08,-0.10,0.19,-0.06,1.00,-0.10,-0.03
latitude,-0.02,0.07,0.18,-0.13,0.18,0.11,-0.24,0.05,-0.10,1.00,0.01


In [18]:
non_numeric_columns = df1.select_dtypes(exclude=[np.number]).columns
print("Non-Numeric Columns:", non_numeric_columns)

label_encoder = LabelEncoder()

for col in non_numeric_columns:
    if col != 'date_time':
        df1[col] = label_encoder.fit_transform(df1[col])

df1['date_time'] = pd.to_datetime(df1['date_time'], dayfirst=True)



Non-Numeric Columns: Index(['title', 'date_time', 'alert', 'net', 'magType', 'location',
       'continent', 'country'],
      dtype='object')


In [19]:
corr = df1.corr()
corr.style.background_gradient(cmap='coolwarm').format("{:.2f}")

,title,magnitude,date_time,cdi,mmi,alert,tsunami,sig,net,nst,dmin,gap,magType,depth,latitude,longitude,location,continent,country
title,1.00,0.91,-0.02,0.16,0.25,0.08,0.02,0.46,-0.06,0.08,-0.07,-0.07,0.04,0.03,-0.04,0.00,0.06,0.01,0.08
magnitude,0.91,1.00,-0.04,0.16,0.28,0.10,-0.00,0.51,-0.09,0.09,-0.08,-0.09,0.04,0.02,-0.02,-0.01,0.03,0.04,0.07
date_time,-0.02,-0.04,1.00,0.62,-0.23,-0.76,0.64,0.19,-0.02,-0.23,0.54,0.42,0.53,0.19,-0.12,-0.13,0.09,0.08,0.17
cdi,0.16,0.16,0.62,1.00,0.20,-0.31,0.29,0.48,-0.07,-0.00,0.14,0.28,0.36,-0.05,0.07,-0.13,-0.02,-0.10,-0.01
mmi,0.25,0.28,-0.23,0.20,1.00,0.33,-0.18,0.40,-0.09,0.11,-0.31,-0.07,-0.09,-0.52,0.18,-0.02,-0.11,-0.39,-0.26
alert,0.08,0.10,-0.76,-0.31,0.33,1.00,-0.70,0.02,-0.05,0.50,-0.59,-0.19,-0.53,-0.24,0.17,0.11,-0.08,-0.14,-0.20
tsunami,0.02,-0.00,0.64,0.29,-0.18,-0.70,1.00,0.02,-0.03,-0.43,0.44,0.24,0.37,0.07,-0.13,-0.13,0.06,0.17,0.24
sig,0.46,0.51,0.19,0.48,0.40,0.02,0.02,1.00,-0.19,0.02,-0.05,0.16,0.07,-0.08,0.18,-0.16,-0.07,-0.21,-0.04
net,-0.06,-0.09,-0.02,-0.07,-0.09,-0.05,-0.03,-0.19,1.00,0.11,0.08,-0.18,0.33,0.05,-0.26,0.25,0.02,0.06,-0.13
nst,0.08,0.09,-0.23,-0.00,0.11,0.50,-0.43,0.02,0.11,1.00,-0.40,0.00,-0.14,-0.10,0.11,0.16,0.00,-0.10,-0.10


In [20]:
features = ['cdi','mmi','tsunami','sig','nst','dmin','gap','depth','latitude','longitude']
target = 'magnitude'

X = df1[features]
y = df1[target]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

print(X_train.dtypes)  # Check the data types of features


cdi            int64
mmi            int64
tsunami        int64
sig            int64
nst            int64
dmin         float64
gap          float64
depth        float64
latitude     float64
longitude    float64
dtype: object


In [21]:
rf_model = RandomForestRegressor(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)

# Make predictions
y_pred = rf_model.predict(X_test)

# Calculate regression evaluation metrics
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"Mean Absolute Error (MAE): {mae:.2f}")
print(f"Mean Squared Error (MSE): {mse:.2f}")
print(f"R-squared (R²): {r2:.2f}")

Mean Absolute Error (MAE): 0.11
Mean Squared Error (MSE): 0.05
R-squared (R²): 0.74
